# Lab 04.6 — Integrated Color Tracker with Zybo LEDs

## Mini-project

Combine the Logitech C270, OpenCV color tracking and the PYNQ base overlay.

The tracked object's horizontal position controls the on-board LEDs:

| Vision result | LED |
|---|---|
| target on LEFT | LED0 |
| target in CENTER | LED1 |
| target on RIGHT | LED2 |
| target LOST | LED3 |

This experiment connects a computer-vision result to physical hardware output.

In [ ]:
import cv2
import numpy as np
import time
from IPython.display import display, Image, clear_output
from pynq import Overlay

ol = Overlay("/home/xilinx/pynq/overlays/base/base.bit")

leds = ol.leds

def all_leds_off():
    for led in leds:
        led.off()

all_leds_off()
print("Base overlay loaded.")

If your PYNQ image exposes the base overlay at a different path, use the same overlay-loading method as in Lab01.

In [ ]:
TARGET = "GREEN"

RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (35, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

def color_mask(hsv, color_name):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in RANGES[color_name]:
        mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

def horizontal_region(cx, width):
    if cx < width / 3:
        return "LEFT"
    elif cx < 2 * width / 3:
        return "CENTER"
    else:
        return "RIGHT"

def set_led_for_region(region):
    all_leds_off()

    if region == "LEFT":
        leds[0].on()
    elif region == "CENTER":
        leds[1].on()
    elif region == "RIGHT":
        leds[2].on()
    else:
        leds[3].on()

## Run the integrated system

Use a large, saturated object of the selected target color.

In [ ]:
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

MIN_AREA = 1200

try:
    for _ in range(240):
        ret, frame = cap.read()
        if not ret:
            print("Camera frame not available.")
            break

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = color_mask(hsv, TARGET)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]

        region = "LOST"

        if valid:
            cnt = max(valid, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(cnt)

            M = cv2.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])

                region = horizontal_region(cx, frame.shape[1])

                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 255, 255), 2)
                cv2.circle(frame, (cx, cy), 6, (255, 255, 255), -1)

        set_led_for_region(region)

        cv2.putText(frame, f"TARGET: {TARGET}  STATE: {region}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 255, 255), 2)

        ok, encoded = cv2.imencode(".jpg", frame)
        if ok:
            clear_output(wait=True)
            display(Image(data=encoded.tobytes()))

        time.sleep(0.03)

finally:
    cap.release()
    all_leds_off()
    print("Camera released. LEDs switched off.")

## Assignment

Extend the system with **one** of the following:

1. **Color selection by switches** — use two switches to select RED, GREEN, BLUE or YELLOW.
2. **Two-object detection** — detect two different colors simultaneously and indicate which one has the larger visible area.
3. **Proximity indicator** — use contour area as a rough indicator of whether the object is FAR, MEDIUM or NEAR.
4. **Event counter** — count how many times the tracked object moves completely from LEFT to RIGHT.

### Discussion

- Which parts of the processing pipeline run on the ARM processor?
- Which operation would you consider moving to FPGA logic in a future hardware-accelerated version?
- What limits the reliability of fixed HSV thresholds in a real environment?